# YogaSafe — Slim Inference Notebook

This is a **minimal inference-only** version of the original training notebook.
It loads the already-trained model from Google Drive, runs video inference,
explains the prediction, and serves it through a FastAPI + Cloudflare tunnel
endpoint. All training, evaluation, and dataset-generation code has been removed.

**Runtime recommendation:** Use a GPU runtime (Runtime → Change runtime type → T4 GPU)
for faster inference. CPU also works, just slower.

**Run order:** run every cell top to bottom, in a single pass, **except** cell 1
(Setup / Installation) — that one requires a runtime restart before you continue
(see the note above it).


## 1. Setup / Installation

Installs the pinned `protobuf`/`mediapipe` versions and downloads the pose
landmarker model.

**⚠️ Run this cell, then restart the runtime (Runtime → Restart session) before
continuing.** This is required because of a MediaPipe/Protobuf version conflict —
this cell must remain isolated from everything else.


In [ ]:
# run lang isang beses
# Step 1: Restore protobuf first
!pip install -q protobuf==5.29.6

# Step 2: Install a mediapipe version built for protobuf 5.x
!pip install -q mediapipe==0.10.21 --no-deps
!pip install -q absl-py>=2.0 flatbuffers>=23.5.26 attrs>=19.1.0

# Step 3: Download pose landmarker model
import urllib.request, os
if not os.path.exists("pose_landmarker.task"):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        "pose_landmarker.task"
    )
    print("Model downloaded")
print("Install done — now restart runtime")
#1

## 2. Imports

Only the libraries needed for inference. Training-only imports
(`matplotlib`, `tqdm`, `sklearn.metrics`, `DataLoader`/`Dataset`) have been removed.

**Restart the runtime first** (per the cell above) before running this.


In [ ]:
# ── Standard library ──
import os
import warnings

warnings.filterwarnings("ignore")

# ── Numerical & Data ──
import numpy as np

# ── Computer Vision & Pose ──
import cv2
import mediapipe as mp

# ── ML / Preprocessing ──
import joblib
from sklearn.preprocessing import RobustScaler, LabelEncoder

# ── Deep Learning ──
import torch
import torch.nn as nn

# ── Timing (used by predict_video) ──
import time

print("All imports successful!")
print(f"MediaPipe: {mp.__version__}")
print(f"PyTorch:   {torch.__version__}")
print(f"Device:    {'cuda' if torch.cuda.is_available() else 'cpu'}")


## 3. Device Selection

Training hyperparameters (`EPOCHS`, `BATCH_SIZE`, `LR`, `LAMBDA_POSE`,
`LAMBDA_RISK`, `GradScaler`, etc.) have been removed — they're not used at inference time.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")


## 4. Biomechanical Feature Extraction

Unchanged from the original notebook.

In [ ]:
## Step 3 – Enhanced Biomechanical Feature Extraction (Scale-Invariant & Unique)
import numpy as np

LM = {"nose": 0, "l_shoulder": 11, "r_shoulder": 12, "l_elbow": 13, "r_elbow": 14, "l_wrist": 15, "r_wrist": 16, "l_hip": 23, "r_hip": 24, "l_knee": 25, "r_knee": 26, "l_ankle": 27, "r_ankle": 28, "l_heel": 29, "r_heel": 30, "l_foot": 31, "r_foot": 32}

def get_xyz(row, name): idx = LM[name]; return row[idx * 3 : idx * 3 + 3]
def angle_3pts(a, b, c):
    v1, v2 = a - b, c - b
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_angle, -1, 1))))

def extract_bio_features_frame(row: np.ndarray) -> dict:
    f = {}
    # Landmarks
    l_sh, r_sh = get_xyz(row, "l_shoulder"), get_xyz(row, "r_shoulder")
    l_el, r_el = get_xyz(row, "l_elbow"), get_xyz(row, "r_elbow")
    l_wr, r_wr = get_xyz(row, "l_wrist"), get_xyz(row, "r_wrist")
    l_hp, r_hp = get_xyz(row, "l_hip"), get_xyz(row, "r_hip")
    l_kn, r_kn = get_xyz(row, "l_knee"), get_xyz(row, "r_knee")
    l_an, r_an = get_xyz(row, "l_ankle"), get_xyz(row, "r_ankle")
    mid_sh, mid_hp = (l_sh + r_sh) / 2, (l_hp + r_hp) / 2

    # Reference scale: Torso Length (Mid-Shoulder to Mid-Hip)
    torso_len = np.linalg.norm(mid_sh[:2] - mid_hp[:2]) + 1e-8

    # 1. Basic Angles (Scale-Invariant by nature)
    f["l_knee_angle"] = angle_3pts(l_hp, l_kn, l_an)
    f["r_knee_angle"] = angle_3pts(r_hp, r_kn, r_an)
    f["l_hip_angle"] = angle_3pts(l_sh, l_hp, l_kn)
    f["r_hip_angle"] = angle_3pts(r_sh, r_hp, r_kn)
    f["l_elbow_angle"] = angle_3pts(l_sh, l_el, l_wr)
    f["r_elbow_angle"] = angle_3pts(r_sh, r_el, r_wr)

    # 2. Normalized Distances (Scale-Invariant Ratios)
    # Tree Specific: Lifted ankle height relative to opposite knee
    # (Positive if one ankle is significantly higher than the other leg's knee)
    l_tree_signal = (r_kn[1] - l_an[1]) / torso_len
    r_tree_signal = (l_kn[1] - r_an[1]) / torso_len
    f["tree_pose_signal"] = float(max(l_tree_signal, r_tree_signal))

    # Crow Specific: Load angle
    f["l_crow_load_angle"] = angle_3pts(l_wr, l_el, l_sh)
    f["r_crow_load_angle"] = angle_3pts(r_wr, r_el, r_sh)

    # Spine / Lumbar
    virtual_lumbar = mid_hp.copy(); virtual_lumbar[2] -= 0.2
    f["lumbar_extension_angle"] = angle_3pts(mid_sh, mid_hp, virtual_lumbar)
    f["lateral_spine_dev_norm"] = float(abs(mid_sh[0] - mid_hp[0]) / torso_len)

    # Stability & Widths (Normalized)
    f["pelvic_tilt_norm"] = float(abs(l_hp[1] - r_hp[1]) / torso_len)
    f["com_height_norm"] = float(((mid_sh[1] + mid_hp[1]) / 2) / torso_len)
    f["stance_width_norm"] = float(np.linalg.norm(l_an[:2] - r_an[:2]) / torso_len)
    f["wrist_distance_norm"] = float(np.linalg.norm(l_wr[:2] - r_wr[:2]) / torso_len)

    return f
    #5

## 5. Sequence Construction

Unchanged from the original notebook. `aggregate_video_features()` references
`augment_landmarks()`, but inference always calls it with `augment=False`, so
that branch never executes and the missing function is never called.


In [ ]:
## Step 4 – Sequence Construction & Video Feature Aggregation

TARGET_FRAMES = 60

def resample_sequence(
    arr: np.ndarray,
    target_len: int = TARGET_FRAMES
):

    n = len(arr)
    if n == target_len:
        return arr
    if n < target_len:
        pad = np.repeat(
            arr[-1:],
            target_len - n,
            axis=0
        )
        return np.concatenate(
            [arr, pad],
            axis=0
        )
    idx = np.linspace(
        0,
        n - 1,
        target_len
    ).astype(int)
    return arr[idx]

def aggregate_video_features(
    frames: np.ndarray,
    pose_label: str,
    risk_label: str,
    video_name: str,
    participant_id: str = None,
    camera_view: str = None,
    augment: bool = False
):
    frame_sources = (
        augment_landmarks(frames)
        if augment
        else [frames]
    )
    results = []
    for src in frame_sources:
        # Per-frame biomechanical features
        bio_rows = [
            extract_bio_features_frame(f)
            for f in src
        ]
        feat_names = list(
            bio_rows[0].keys()
        )
        bio_arr = np.array(
            [
                list(r.values())
                for r in bio_rows
            ]
        )

        # Transformer Sequences
        landmark_sequence = resample_sequence(
            src,
            TARGET_FRAMES
        )
        biomech_sequence = resample_sequence(
            bio_arr,
            TARGET_FRAMES
        )
        # ==========================================
        # Aggregated Statistics
        # ==========================================
        agg = {}
        for j, name in enumerate(feat_names):
            col = bio_arr[:, j]
            agg[f"{name}_mean"] = float(
                np.mean(col)
            )
            agg[f"{name}_std"] = float(
                np.std(col)
            )
            agg[f"{name}_min"] = float(
                np.min(col)
            )
            agg[f"{name}_max"] = float(
                np.max(col)
            )
            agg[f"{name}_range"] = float(
                np.ptp(col)
            )
        # Stability Metrics
        xy = src[:, :66].reshape(
            len(src),
            33,
            2
        )

        agg["position_variance"] = float(
            np.mean(
                np.var(
                    xy,
                    axis=0
                )
            )
        )
        if len(src) >= 3:
            vel = np.diff(
                xy,
                axis=0
            )
            accel = np.diff(
                vel,
                axis=0
            )
            agg["jerk_magnitude"] = float(
                np.mean(
                    np.linalg.norm(
                        accel.reshape(
                            len(accel),
                            -1
                        ),
                        axis=1
                    )
                )
            )

        else:
            agg["jerk_magnitude"] = 0.0
        angle_features = [
            i
            for i, n in enumerate(feat_names)
            if "angle" in n
        ]
        if len(angle_features) > 0:

            angle_arr = bio_arr[
                :,
                angle_features
            ]
            agg["tremor_index"] = float(
                np.mean(
                    np.std(
                        angle_arr,
                        axis=0
                    )
                )
            )

        else:
            agg["tremor_index"] = 0.0
        # Metadata

        agg["video_name"] = video_name
        agg["pose_label"] = pose_label
        agg["risk_label"] = risk_label
        agg["participant_id"] = participant_id
        agg["camera_view"] = camera_view
        # Store sequences for Transformer
        agg["landmark_sequence"] = landmark_sequence.astype(np.float32)
        agg["biomech_sequence"] = biomech_sequence.astype(np.float32)

        results.append(agg)

    return results
print("Sequence aggregation function defined.")
print(f"Target sequence length = {TARGET_FRAMES}")
print("Transformer-compatible output enabled.")
#6

## 6. Model Architecture

`PositionalEncoding` and `MultiTaskYogaTransformer`, unchanged from the original.

Must run **before** the checkpoint-restore cell below.

> Note: the original notebook's trailing print statement referenced `NUM_JOINTS`,
> which is only defined later in the restore step (it used to come from an
> earlier training cell that's no longer part of this slim notebook). That
> print line was removed here to avoid a `NameError` — no functional code was changed.


In [ ]:
# ── FIX 2: Model with joint-gated risk head ───────────────────────────────────
# The key change: joint_risk_head now *informs* risk_head via a learned gate.
# joint logits are projected to a gate vector and multiplied element-wise into
# the fused representation before the risk classifier reads it.
# This lets the model say "knee angle is risky → amplify knee-related features
# in the fusion vector before making the safe/unsafe call".

import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(weights * x, dim=1)


class MultiTaskYogaTransformer(nn.Module):
    """
    Three-head transformer with joint-gated risk head:
      1. pose_head       → pose classification      (CrossEntropyLoss)
      2. risk_head       → safe/unsafe              (CrossEntropyLoss w/ class weights)
      3. joint_risk_head → per-joint risk scores    (BCEWithLogitsLoss)

    NEW: joint logits gate the fused vector before risk_head reads it.
    joint_gate = sigmoid(Linear(joint_logits)) → element-wise scale of fused.
    This makes joint_risk_head directly causal for the risk decision.
    """
    def __init__(self, landmark_dim, biomech_dim, agg_dim,
                 num_poses, num_risks, num_joints,
                 d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super().__init__()

        # ── Landmark branch ──────────────────────────────────────────────────
        self.landmark_proj    = nn.Linear(landmark_dim, d_model)
        self.landmark_pos     = PositionalEncoding(d_model)
        lm_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.landmark_encoder = nn.TransformerEncoder(lm_layer, num_layers=num_layers)
        self.landmark_pool    = AttentionPooling(d_model)

        # ── Biomechanical branch ──────────────────────────────────────────────
        self.biomech_proj    = nn.Linear(biomech_dim, d_model)
        self.biomech_pos     = PositionalEncoding(d_model)
        bio_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.biomech_encoder = nn.TransformerEncoder(bio_layer, num_layers=num_layers)
        self.biomech_pool    = AttentionPooling(d_model)

        # ── Aggregated stats branch ───────────────────────────────────────────
        self.agg_proj = nn.Sequential(
            nn.Linear(agg_dim, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, d_model)
        )

        # ── Fusion ────────────────────────────────────────────────────────────
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # ── Output heads ──────────────────────────────────────────────────────
        self.pose_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, num_poses)
        )

        # Joint risk head (unchanged interface, same loss)
        self.joint_risk_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_joints)
        )

        # ── NEW: joint gate projects joint logits → d_model scale vector ──────
        # sigmoid output → 0..1, used as element-wise multiplicative gate.
        # This makes joint predictions directly modulate the fused features
        # before risk_head reads them.
        self.joint_gate = nn.Sequential(
            nn.Linear(num_joints, d_model),
            nn.Sigmoid()                        # gate values in [0, 1]
        )

        # Risk head reads the gated fused vector
        self.risk_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, num_risks)
        )

    def forward(self, landmark_seq, biomech_seq, agg_feat):
        # Encode each branch
        l_feat = self.landmark_pool(
            self.landmark_encoder(
                self.landmark_pos(self.landmark_proj(landmark_seq))
            )
        )
        b_feat = self.biomech_pool(
            self.biomech_encoder(
                self.biomech_pos(self.biomech_proj(biomech_seq))
            )
        )
        a_feat = self.agg_proj(agg_feat)

        # Fuse
        fused = self.fusion(torch.cat([l_feat, b_feat, a_feat], dim=1))

        # Pose head (reads fused directly)
        pose_logits = self.pose_head(fused)

        # Joint head (reads fused, produces per-joint logits)
        joint_logits = self.joint_risk_head(fused)

        # ── NEW: gate fused with joint evidence before risk decision ──────────
        gate = self.joint_gate(joint_logits)    # (B, d_model), values in [0,1]
        gated_fused = fused * gate              # amplify / suppress features

        # Risk head (reads gated fused)
        risk_logits = self.risk_head(gated_fused)

        return pose_logits, risk_logits, joint_logits


print("MultiTaskYogaTransformer (joint-gated risk head) defined ✓")

## 7. Restore Model from Google Drive

Mounts Google Drive, copies `YogaModel_SavedArtifacts` locally, loads the
checkpoint + scalers + feature columns, rebuilds the label encoders and
`JOINT_NAMES`, reconstructs `MultiTaskYogaTransformer`, and loads the trained
weights into `final_model`. This is the primary restore step — no training or
data-loading cells run before this.

**Builder note — artifact copy strategy:** this copies the *entire*
`YogaModel_SavedArtifacts` folder (landmarks, features, sequences, and models),
matching the original notebook's behavior, since inference only strictly needs
the `models/` subfolder. If Drive storage or startup time is a concern, you can
change `shutil.copytree(DRIVE_ARTIFACTS, OUTPUT_ROOT)` to copy just
`DRIVE_ARTIFACTS/models` into `OUTPUT_ROOT/models` for a faster startup.

**Builder note — verify the Drive path:** confirm that
`/content/drive/MyDrive/YogaModel_SavedArtifacts` exists and contains your
latest saved artifacts before running this cell.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔄  RESTORE FROM GOOGLE DRIVE  (run this after a session reset)
# Run AFTER: the Imports cell, the Device Selection cell, and the Model Architecture cell above.
# Skip ALL training / data-loading cells — go straight to FastAPI
# ══════════════════════════════════════════════════════════════

import os, shutil, joblib, torch
import numpy as np
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# ── 1. Mount Drive ────────────────────────────────────────────
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

# ── 2. Restore artifact folder from Drive → /content/ ────────
DRIVE_ARTIFACTS = "/content/drive/MyDrive/YogaModel_SavedArtifacts"
OUTPUT_ROOT     = "/content/YogaModel_SavedArtifacts"

if not os.path.exists(OUTPUT_ROOT):
    print(f"Copying artifacts from Drive …")
    shutil.copytree(DRIVE_ARTIFACTS, OUTPUT_ROOT)
    print("Done.")
else:
    print("Artifacts already present locally, skipping copy.")

MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")

# ── 3. Load scalers + feature_cols ───────────────────────────
feature_scaler  = joblib.load(os.path.join(MODEL_DIR, "bio_feature_scaler.pkl"))
landmark_scaler = joblib.load(os.path.join(MODEL_DIR, "landmark_scaler.pkl"))
biomech_scaler  = joblib.load(os.path.join(MODEL_DIR, "biomech_scaler.pkl"))
feature_cols    = joblib.load(os.path.join(MODEL_DIR, "bio_feature_cols.pkl"))
print(f"Scalers loaded  |  feature_cols: {len(feature_cols)}")

# ── 4. Load model checkpoint ──────────────────────────────────
MODEL_PATH = os.path.join(MODEL_DIR, "multitask_yoga_transformer.pth")
ckpt = torch.load(MODEL_PATH, map_location=device)

# ── 5. Rebuild label encoders from checkpoint metadata ────────
pose_encoder = LabelEncoder()
risk_encoder = LabelEncoder()
pose_encoder.classes_ = np.array(ckpt["pose_classes"])
risk_encoder.classes_ = np.array(ckpt["risk_classes"])
JOINT_NAMES = ckpt["joint_names"]
NUM_JOINTS  = len(JOINT_NAMES)
print(f"Pose classes : {list(pose_encoder.classes_)}")
print(f"Risk classes : {list(risk_encoder.classes_)}")
print(f"Joint names  : {JOINT_NAMES}")

# ── 6. Rebuild + load the model ───────────────────────────────
final_model = MultiTaskYogaTransformer(
    landmark_dim = ckpt["landmark_dim"],
    biomech_dim  = ckpt["biomech_dim"],
    agg_dim      = ckpt["agg_dim"],
    num_poses    = ckpt["num_poses"],
    num_risks    = ckpt["num_risks"],
    num_joints   = ckpt["num_joints"],
    d_model      = 256,
    nhead        = 8,
    num_layers   = 4,
    dropout      = 0.30,
).to(device)

final_model.load_state_dict(ckpt["model_state_dict"])
final_model.eval()
print("✅  final_model loaded and ready for inference.")

## 8. Inference

`predict_video()` — unchanged from the original notebook.

In [ ]:
## Step 14 – predict_video (3-Head Inference)

def predict_video(video_path):
    timings = {}
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    t0 = time.perf_counter()
    mp_pose = mp.solutions.pose
    landmark_rows = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"error": f"Could not open video: {video_path}"}
    with mp_pose.Pose(static_image_mode=False, model_complexity=2) as detector:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = detector.process(frame_rgb)
            if result.pose_landmarks:
                row = []
                for lm in result.pose_landmarks.landmark:
                    row.extend([lm.x, lm.y, lm.z])
                landmark_rows.append(row)
    cap.release()
    timings["landmark_extraction_ms"] = (time.perf_counter() - t0) * 1000

    if len(landmark_rows) < 5:
        return {"error": "Video too short or no landmarks detected."}

    frames = np.array(landmark_rows, dtype=np.float32)

    t1 = time.perf_counter()
    agg_results_list = aggregate_video_features(
        frames=frames, pose_label="tmp", risk_label="tmp", video_name="tmp", augment=False
    )
    agg_results = agg_results_list[0]
    timings["bio_feature_extraction_ms"] = (time.perf_counter() - t1) * 1000

    t2 = time.perf_counter()
    agg_raw    = np.array([agg_results[k] for k in feature_cols]).astype(np.float32)
    agg_scaled = feature_scaler.transform(agg_raw.reshape(1, -1))
    l_seq      = agg_results["landmark_sequence"]
    l_dim      = l_seq.shape[-1]
    l_seq_scaled = landmark_scaler.transform(l_seq.reshape(-1, l_dim)).reshape(1, TARGET_FRAMES, l_dim)
    b_seq      = agg_results["biomech_sequence"]
    b_dim      = b_seq.shape[-1]
    b_seq_scaled = biomech_scaler.transform(b_seq.reshape(-1, b_dim)).reshape(1, TARGET_FRAMES, b_dim)
    timings["preprocessing_ms"] = (time.perf_counter() - t2) * 1000

    t3 = time.perf_counter()
    l_tensor = torch.tensor(l_seq_scaled, dtype=torch.float32).to(device)
    b_tensor = torch.tensor(b_seq_scaled, dtype=torch.float32).to(device)
    a_tensor = torch.tensor(agg_scaled,   dtype=torch.float32).to(device)
    timings["tensor_transfer_ms"] = (time.perf_counter() - t3) * 1000

    final_model.eval()
    with torch.no_grad():
        _ = final_model(l_tensor, b_tensor, a_tensor)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t4 = time.perf_counter()
    with torch.no_grad():
        for _ in range(10):
            p_logits, r_logits, j_logits = final_model(l_tensor, b_tensor, a_tensor)
    if device.type == "cuda":
        torch.cuda.synchronize()
    timings["model_inference_ms"] = (time.perf_counter() - t4) * 1000 / 10

    t5 = time.perf_counter()
    p_prob = torch.softmax(p_logits, dim=1)
    r_prob = torch.softmax(r_logits, dim=1)
    p_idx  = p_prob.argmax(1).item()
    r_idx  = r_prob.argmax(1).item()
    timings["postprocessing_ms"] = (time.perf_counter() - t5) * 1000
    timings["total_ms"] = sum(timings.values())

    if device.type == "cuda":
        timings["gpu_memory_allocated_mb"] = torch.cuda.memory_allocated(device) / 1e6
        timings["gpu_memory_reserved_mb"]  = torch.cuda.memory_reserved(device) / 1e6

    return {
        "Predicted Pose"   : pose_encoder.classes_[p_idx],
        "Pose Confidence"  : f"{p_prob[0][p_idx].item()*100:.2f}%",
        "Predicted Risk"   : risk_encoder.classes_[r_idx],
        "Risk Confidence"  : f"{r_prob[0][r_idx].item()*100:.2f}%",
        "Frames Processed" : len(landmark_rows),
        "Device"           : str(device),
        "timings"          : timings
    }

print("predict_video defined (3-head) ✓")


## 9. Explainability

`explain_risk()` and `integrated_gradients_agg()`, plus the supporting display
names, units, and safe-range dictionaries used for reporting. Unchanged from
the original notebook.


In [ ]:
## Step 15 – explain_risk() — Model-Driven Joint Localization + IG Support + Unsafe Signal Decomposition

# DISPLAY NAMES for joint/feature keys
JOINT_DISPLAY_NAMES = {
    "l_knee_angle"           : "Left knee angle",
    "r_knee_angle"           : "Right knee angle",
    "l_hip_angle"            : "Left hip angle",
    "r_hip_angle"            : "Right hip angle",
    "l_elbow_angle"          : "Left elbow angle",
    "r_elbow_angle"          : "Right elbow angle",
    "l_crow_load_angle"      : "Left wrist/elbow load",
    "r_crow_load_angle"      : "Right wrist/elbow load",
    "lumbar_extension_angle" : "Lumbar spine angle",
    "lateral_spine_dev_norm" : "Lateral spine lean",
    "pelvic_tilt_norm"       : "Pelvic tilt",
    "stance_width_norm"      : "Stance width",
    "tree_pose_signal"       : "Tree pose leg lift",
}

JOINT_UNITS = {
    "l_knee_angle": "°", "r_knee_angle": "°",
    "l_hip_angle":  "°", "r_hip_angle":  "°",
    "l_elbow_angle":"°", "r_elbow_angle":"°",
    "l_crow_load_angle": "°", "r_crow_load_angle": "°",
    "lumbar_extension_angle": "°",
}

# Safe ranges — kept here for DISPLAY/reporting only, not for inference decisions
SAFE_RANGES_DISPLAY = {
    "l_knee_angle"           : (100, 170),
    "r_knee_angle"           : (100, 170),
    "l_hip_angle"            : (60,  160),
    "r_hip_angle"            : (60,  160),
    "l_elbow_angle"          : (30,  170),
    "r_elbow_angle"          : (30,  170),
    "l_crow_load_angle"      : (30,  160),
    "r_crow_load_angle"      : (30,  160),
    "lumbar_extension_angle" : (140, 200),
    "lateral_spine_dev_norm" : (0,   0.15),
    "pelvic_tilt_norm"       : (0,   0.12),
    "stance_width_norm"      : (0,   1.8),
    "tree_pose_signal"       : (-0.5, 0.8),
}

STAT_WEIGHTS = {"_mean": 1.0, "_max": 0.6, "_min": 0.3, "_range": 0.5, "_std": 0.4}


def integrated_gradients_agg(model, l_tensor, b_tensor, a_tensor, target_class, steps=50):
    """
    Integrated Gradients on the aggregated feature vector for the risk head.
    Returns attributions of shape (n_features,) — positive = pushed toward unsafe.
    """
    model.eval()
    baseline = torch.zeros_like(a_tensor)
    alphas   = torch.linspace(0, 1, steps).to(a_tensor.device)
    grads    = []
    for alpha in alphas:
        inp = (baseline + alpha * (a_tensor - baseline)).detach().requires_grad_(True)
        _, r_logits, _ = model(l_tensor, b_tensor, inp)
        r_logits[0, target_class].backward()
        grads.append(inp.grad.detach().cpu().numpy().copy())
    grads     = np.array(grads)
    avg_grads = grads.mean(axis=0)[0]
    delta     = (a_tensor - baseline).detach().cpu().numpy()[0]
    return avg_grads * delta


def explain_risk(video_path, model=None, verbose=True):
    """
    Runs the full pipeline and explains which joints the model flagged as risky.

    Joint localization comes from TWO model-internal sources:
      1. joint_risk_head: sigmoid probabilities per joint (learned from training labels).
         This is the PRIMARY localization — the model itself predicts which joints are at risk.
      2. Integrated Gradients on agg features: attribution of each feature to the
         safe/unsafe decision. Used as a SUPPORTING signal to confirm the joint ranking.

    Unsafe signal decomposition:
      Even when the model predicts SAFE, the residual unsafe probability (e.g. 4.5%)
      is decomposed per joint using:
          joint_contribution[j] = joint_prob[j] × unsafe_prob
      This shows the panel exactly which joints account for the model's remaining doubt.

    The SAFE_RANGES_DISPLAY values are shown for human context only — they do NOT
    drive the ranking or the primary cause determination.
    """
    if model is None:
        model = final_model

    device_local  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mp_pose       = mp.solutions.pose
    landmark_rows = []

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"error": f"Could not open video: {video_path}"}

    with mp_pose.Pose(static_image_mode=False, model_complexity=2) as detector:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result    = detector.process(frame_rgb)
            if result.pose_landmarks:
                row = []
                for lm in result.pose_landmarks.landmark:
                    row.extend([lm.x, lm.y, lm.z])
                landmark_rows.append(row)
    cap.release()

    if len(landmark_rows) < 5:
        return {"error": "Video too short or no landmarks detected."}

    frames      = np.array(landmark_rows, dtype=np.float32)
    agg_list    = aggregate_video_features(frames, "tmp", "tmp", "tmp", augment=False)
    agg_results = agg_list[0]

    agg_raw    = np.array([agg_results[k] for k in feature_cols], dtype=np.float32)
    agg_scaled = feature_scaler.transform(agg_raw.reshape(1, -1))

    l_seq = agg_results["landmark_sequence"]; l_dim = l_seq.shape[-1]
    b_seq = agg_results["biomech_sequence"];  b_dim = b_seq.shape[-1]
    l_seq_scaled = landmark_scaler.transform(l_seq.reshape(-1, l_dim)).reshape(1, TARGET_FRAMES, l_dim)
    b_seq_scaled = biomech_scaler.transform(b_seq.reshape(-1, b_dim)).reshape(1, TARGET_FRAMES, b_dim)

    l_tensor = torch.tensor(l_seq_scaled, dtype=torch.float32).to(device_local)
    b_tensor = torch.tensor(b_seq_scaled, dtype=torch.float32).to(device_local)
    a_tensor = torch.tensor(agg_scaled,   dtype=torch.float32).to(device_local)

    # ── Model prediction ──────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        p_logits, r_logits, j_logits = model(l_tensor, b_tensor, a_tensor)

    p_prob = torch.softmax(p_logits, dim=1)
    r_prob = torch.softmax(r_logits, dim=1)
    p_idx  = p_prob.argmax(1).item()
    r_idx  = r_prob.argmax(1).item()
    predicted_pose = pose_encoder.classes_[p_idx]
    predicted_risk = risk_encoder.classes_[r_idx]

    # ── SOURCE 1: joint_risk_head (model-predicted per-joint risk) ────────────
    joint_probs = torch.sigmoid(j_logits[0]).cpu().detach().numpy()  # (NUM_JOINTS,)

    # ── SOURCE 2: Integrated Gradients (supporting signal) ────────────────────
    ig_scores = integrated_gradients_agg(model, l_tensor, b_tensor, a_tensor,
                                         target_class=r_idx, steps=50)
    ig_per_joint = {j: 0.0 for j in JOINT_NAMES}
    for i, col_name in enumerate(feature_cols):
        for base in JOINT_NAMES:
            for suffix, w in STAT_WEIGHTS.items():
                if col_name == base + suffix:
                    ig_per_joint[base] += float(ig_scores[i]) * w
                    break

    # ── Raw values for display ─────────────────────────────────────────────────
    joint_values = {}
    for base in JOINT_NAMES:
        mean_key = base + "_mean"
        for i, col in enumerate(feature_cols):
            if col == mean_key:
                joint_values[base] = float(agg_raw[i])
                break

    # ── Unsafe signal decomposition ────────────────────────────────────────────
    # Find which class index is "unsafe"
    risk_classes    = list(risk_encoder.classes_)
    unsafe_class_idx = risk_classes.index("unsafe") if "unsafe" in risk_classes else (1 - r_idx)
    safe_class_idx   = risk_classes.index("safe")   if "safe"   in risk_classes else r_idx

    safe_prob   = float(r_prob[0][safe_class_idx].item())    # e.g. 0.955
    unsafe_prob = float(r_prob[0][unsafe_class_idx].item())  # e.g. 0.045

    # Each joint's raw contribution to the unsafe probability:
    #   contribution[j] = joint_prob[j] × unsafe_prob
    # This answers: "of the X% unsafe signal, how much comes from this joint?"
    joint_raw_contrib = {
        JOINT_NAMES[j]: float(joint_probs[j]) * unsafe_prob
        for j in range(NUM_JOINTS)
    }

    # Normalise so contributions sum exactly to unsafe_prob, expressed as percentage points
    total_raw = sum(joint_raw_contrib.values())
    joint_contrib_pp = {
        k: (v / total_raw) * unsafe_prob * 100   # percentage points (pp)
        for k, v in joint_raw_contrib.items()
    }

    # Sorted for display
    sorted_contrib = sorted(joint_contrib_pp.items(), key=lambda x: x[1], reverse=True)

    # ── Build full explanation ranked by MODEL joint probability (primary) ─────
    ranked_joints = sorted(
        enumerate(JOINT_NAMES),
        key=lambda x: joint_probs[x[0]],
        reverse=True
    )

    explanation = []
    for jidx, joint_name in ranked_joints:
        display  = JOINT_DISPLAY_NAMES.get(joint_name, joint_name.replace("_", " ").title())
        prob     = float(joint_probs[jidx])
        ig_score = ig_per_joint.get(joint_name, 0.0)
        val      = joint_values.get(joint_name)
        unit     = JOINT_UNITS.get(joint_name, "")
        safe_r   = SAFE_RANGES_DISPLAY.get(joint_name)
        oor      = (not (safe_r[0] <= val <= safe_r[1])) if (safe_r and val is not None) else False
        contrib  = joint_contrib_pp.get(joint_name, 0.0)

        explanation.append({
            "body_part"        : display,
            "feature_key"      : joint_name,
            "model_prob"       : round(prob, 4),
            "ig_score"         : round(ig_score, 4),
            "unsafe_contrib_pp": round(contrib, 4),   # percentage points of total unsafe prob
            "value"            : round(val, 2) if val is not None else None,
            "unit"             : unit,
            "safe_range"       : safe_r,
            "out_of_range"     : oor,
        })

    primary_cause = explanation[0]["body_part"] if explanation else "Unknown"

    # ── Print report ───────────────────────────────────────────────────────────
    if verbose:
        print("=" * 72)
        print("YOGASAFE RISK EXPLANATION REPORT")
        print("=" * 72)
        print(f"  Video           : {os.path.basename(video_path)}")
        print(f"  Predicted Pose  : {predicted_pose}  ({p_prob[0][p_idx].item()*100:.1f}% confidence)")
        print(f"  Predicted Risk  : {predicted_risk.upper()}  ({r_prob[0][r_idx].item()*100:.1f}% confidence)")
        print()

        # ── Section 1: Joint risk ranking ─────────────────────────────────────
        print("  ─── JOINT RISK ASSESSMENT (ranked by model-predicted probability) ───")
        print("  Source: joint_risk_head — learned from labeled training data")
        print()

        for i, e in enumerate(explanation[:8], 1):
            val_str  = f"{e['value']}{e['unit']}" if e["value"] is not None else "N/A"
            oor_note = "  [outside reference range]" if e["out_of_range"] else ""
            ig_dir   = "↑ risky" if e["ig_score"] > 0 else "↓ safe"
            bar_len  = int(e["model_prob"] * 20)
            bar      = "█" * bar_len + "░" * (20 - bar_len)
            print(f"  #{i:<2} {e['body_part']:<30}")
            print(f"       Model prob        : {e['model_prob']:.4f}  [{bar}]")
            print(f"       IG support        : {e['ig_score']:+.4f}  {ig_dir}")
            print(f"       Unsafe contrib    : {e['unsafe_contrib_pp']:.4f}pp  (of {unsafe_prob*100:.2f}% total unsafe)")
            print(f"       Value             : {val_str}{oor_note}")
            if e["safe_range"]:
                lo, hi = e["safe_range"]; unit = e["unit"]
                print(f"       Ref range         : {lo}{unit} – {hi}{unit}  (annotation reference)")
            print()

        print(f"  PRIMARY CAUSE (model-identified) : {primary_cause}")
        print(f"  Model probability of risk         : {explanation[0]['model_prob']:.4f}")
        print()

        # ── Section 2: Unsafe signal decomposition ────────────────────────────
        print(f"  ─── WHERE DOES THE {unsafe_prob*100:.2f}% UNSAFE SIGNAL COME FROM? ───")
        print(f"  Even though predicted {predicted_risk.upper()} ({safe_prob*100:.2f}% confidence),")
        print(f"  the remaining {unsafe_prob*100:.2f}% unsafe probability is explained by:")
        print()

        top5_total = 0.0
        for joint_key, contrib_pp in sorted_contrib[:5]:
            display  = JOINT_DISPLAY_NAMES.get(joint_key, joint_key.replace("_", " ").title())
            share    = (contrib_pp / (unsafe_prob * 100)) * 100  # % share of the unsafe signal
            bar_len  = int(share / 5)                             # scale: 100% share = 20 blocks
            bar      = "█" * bar_len + "░" * (20 - bar_len)
            print(f"  {display:<30}  {contrib_pp:.4f}pp  [{bar}]  ({share:.1f}% of unsafe signal)")
            top5_total += contrib_pp

        print()
        print(f"  Top-5 joints account for : {top5_total:.4f}pp of {unsafe_prob*100:.2f}%")
        print(f"  Remaining {len(sorted_contrib)-5} joints account for : {(unsafe_prob*100 - top5_total):.4f}pp")
        print()
        print("  Interpretation:")
        top_joint_display = JOINT_DISPLAY_NAMES.get(sorted_contrib[0][0], sorted_contrib[0][0])
        print(f"    The model's residual doubt is driven primarily by {top_joint_display}.")
        print(f"    This joint has the highest learned risk probability AND contributes")
        print(f"    the most to the {unsafe_prob*100:.2f}% of the model's unsafe signal.")
        print()
        print("  Note: Rankings are determined entirely by the joint_risk_head output.")
        print("  Integrated Gradients scores provide a secondary interpretability check.")
        print("  Reference ranges shown for human context — not used during inference.")
        print("=" * 72)

    return {
        "Predicted Pose"    : predicted_pose,
        "Pose Confidence"   : f"{p_prob[0][p_idx].item()*100:.2f}%",
        "Predicted Risk"    : predicted_risk,
        "Risk Confidence"   : f"{r_prob[0][r_idx].item()*100:.2f}%",
        "Safe Probability"  : round(safe_prob * 100, 4),
        "Unsafe Probability": round(unsafe_prob * 100, 4),
        "Frames Processed"  : len(landmark_rows),
        "explanation"       : explanation,
        "primary_cause"     : primary_cause,
        "unsafe_decomposition": dict(sorted_contrib),
    }

print("explain_risk() defined — model-driven joint localization + unsafe decomposition ✓")
print("Usage: result = explain_risk('/content/crow.mp4')")


## 10. API Dependencies

Installs FastAPI, uvicorn, and cloudflared.

In [ ]:
import os
# Install FastAPI and Cloudflare tunnel requirements
!pip install -q fastapi uvicorn python-multipart nest-asyncio

In [ ]:
# Install cloudflared
print("Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("Cloudflared installation complete.")

## 11. FastAPI Server + Cloudflare Tunnel

Serves `/predict` and `/health` via a background uvicorn server, tunneled
through `cloudflared`. Prints the public `https://xxxxx.trycloudflare.com`
URL for website integration.

**Builder note — naming:** the original notebook used `app2` / port `8001` to
avoid colliding with another server running alongside it during training. Since
this slim notebook runs standalone, it's been renamed back to `app` / port `8000`.


In [ ]:


import nest_asyncio, uvicorn, tempfile, os, subprocess, threading, time, re
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    suffix = os.path.splitext(file.filename)[-1] or ".mp4"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name
    try:
        # Step 1: predict_video → pose, risk, timings
        pred = predict_video(tmp_path)
        if "error" in pred:
            return pred

        # Step 2: explain_risk → joint probs, safe/unsafe split, decomp
        explain = explain_risk(tmp_path, model=final_model, verbose=False)
        if "error" in explain:
            return {**pred, "explain_error": explain["error"]}

        # Merge into one response
        return {
            "Predicted Pose"      : pred.get("Predicted Pose"),
            "Pose Confidence"     : pred.get("Pose Confidence"),
            "Predicted Risk"      : pred.get("Predicted Risk"),
            "Risk Confidence"     : pred.get("Risk Confidence"),
            "Frames Processed"    : pred.get("Frames Processed"),
            "Device"              : pred.get("Device"),
            "timings"             : pred.get("timings"),
            # explain_risk fields — feed the HTML report
            "Safe Probability"    : explain.get("Safe Probability"),
            "Unsafe Probability"  : explain.get("Unsafe Probability"),
            "explanation"         : explain.get("explanation"),
            "primary_cause"       : explain.get("primary_cause"),
            "unsafe_decomposition": explain.get("unsafe_decomposition"),
        }
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

@app.get("/health")
def health():
    return {"status": "ok"}

# Slim notebook runs standalone, so we use the default port 8000
def start_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=start_uvicorn, daemon=True).start()
time.sleep(1)

print("Starting Cloudflare Tunnel on port 8000...")
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

for line in proc.stdout:
    decoded_line = line.decode()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", decoded_line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 60)
        print("YOGASAFE BACKEND IS LIVE (port 8000)!")
        print(f"API URL: {url}")
        print("Paste this URL into your HTML site.")
        print("=" * 60)
        break

## Instructions for Running the Notebook

1. **Run Section 1 (Setup / Installation)**, then **restart the runtime**
   (Runtime → Restart session). This is the only cell that needs isolation.
2. After restarting, run **Sections 2–11 in order, top to bottom**, in a single pass:
   - Imports → Device Selection → Biomechanical Feature Extraction →
     Sequence Construction → Model Architecture → Restore Model from Google
     Drive → Inference → Explainability → API Dependencies → FastAPI Server.
3. Once the FastAPI server cell finishes starting, watch the output for:
   ```
   YOGASAFE BACKEND IS LIVE (port 8000)!
   API URL: https://xxxxx.trycloudflare.com
   ```
   Paste that URL into your website/front-end integration.
4. To run inference directly inside the notebook instead of (or in addition
   to) the API, call:
   ```python
   result = predict_video("/content/your_video.mp4")
   explanation = explain_risk("/content/your_video.mp4")
   ```

### What's different from the original training notebook
- All dataset creation, preprocessing-for-training, model training, evaluation,
  and artifact-saving cells have been removed — the trained model already
  exists in Google Drive, so none of that belongs in an inference notebook.
- Training-only imports and hyperparameters were trimmed.
- The FastAPI app/port were renamed from `app2`/`8001` back to `app`/`8000`
  since this notebook no longer needs to coexist with another server.

### Runtime recommendation
Use a **GPU runtime** for faster inference. CPU is fully supported, just slower.
